In [31]:
import numpy as np
import tensorflow as tf 
from tensorflow import keras 
from tensorflow.keras import layers 

In [32]:
(X_train,_),(X_test,_) = tf.keras.datasets.fashion_mnist.load_data()

In [33]:
X_train = X_train.astype('float32')/255
X_test = X_test.astype('float32')/255

In [34]:
X_train = np.reshape(X_train, (-1,28,28,1))
X_test = np.reshape(X_train, (-1,28,28,1))

In [35]:
#build encoder
latend_dim = 2

encoder_input = layers.Input(shape=(28,28,1))
x = layers.Flatten()(encoder_input)
x = layers.Dense(128, activation='relu')(x)

z_mean = layers.Dense(latend_dim, name='z_mean')(x)

z_log_var = layers.Dense(latend_dim, name='z_log_var')(x)

In [36]:
def sampling(args):
    z_mean,z_log_var = args
    epsilon = tf.random.normal(shape=tf.shape(z_mean))
    z = z_mean + tf.exp(0.5*z_log_var)*epsilon
    return z 
z = layers.Lambda(sampling)([z_mean,z_log_var])

In [37]:
latend_input = keras.Input(shape=(latend_dim,))
x = layers.Dense(128,activation='relu')(latend_input)
x = layers.Dense(28*28, activation='sigmoid')(x)
decoder_ouput = layers.Reshape((28,28,1))(x)
decoder = keras.Model(latend_input, decoder_ouput, name='decoder')


In [38]:
output = decoder(z)
vae = keras.Model(encoder_input,output)

In [39]:
reconstruction_loss = keras.losses.binary_crossentropy(
    encoder_input,
    output
)

reconstruction_loss = tf.reduce_sum(
    reconstruction_loss,
    axis=(1, 2)
)

kl_loss = -0.5 * tf.reduce_sum(
    1 + z_log_var
    - tf.square(z_mean)
    - tf.exp(z_log_var),
    axis=1
)

vae.add_loss(
    tf.reduce_mean(reconstruction_loss + kl_loss)
)

vae.compile(optimizer='adam')

In [41]:
history = vae.fit(
    X_train,
    epochs=10,
    batch_size=128,
    validation_data=(X_test, None)
)

Epoch 1/10


StagingError: in user code:

    File "c:\Users\MINA\Documents\WeekilyPrac\Ml_JOUNRAL\ml-learning-journal\GEN AI\.venv\lib\site-packages\keras\engine\training.py", line 1160, in train_function  *
        return step_function(self, iterator)
    File "c:\Users\MINA\Documents\WeekilyPrac\Ml_JOUNRAL\ml-learning-journal\GEN AI\.venv\lib\site-packages\keras\engine\training.py", line 1146, in step_function  **
        outputs = model.distribute_strategy.run(run_step, args=(data,))
    File "c:\Users\MINA\Documents\WeekilyPrac\Ml_JOUNRAL\ml-learning-journal\GEN AI\.venv\lib\site-packages\keras\engine\training.py", line 1135, in run_step  **
        outputs = model.train_step(data)
    File "c:\Users\MINA\Documents\WeekilyPrac\Ml_JOUNRAL\ml-learning-journal\GEN AI\.venv\lib\site-packages\keras\engine\training.py", line 993, in train_step
        y_pred = self(x, training=True)
    File "c:\Users\MINA\Documents\WeekilyPrac\Ml_JOUNRAL\ml-learning-journal\GEN AI\.venv\lib\site-packages\keras\util